In [11]:
import torch
import psutil

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB
RAM: 179.4 GB


In [12]:
!pip install transformers timm huggingface_hub scikit-learn tqdm -q
print("Installation complete")

Installation complete


In [13]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('HuggingFace Login OK!')

HuggingFace Login OK!


In [14]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/clip/pcam'
RESULTS_DIR = f'{BASE_DIR}/results'
#os.makedirs(EMBED_DIR, exist_ok=True)
#os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Directories ready: {BASE_DIR}")

Mounted at /content/drive
Directories ready: /content/drive/MyDrive/PRISM


In [15]:
from transformers import CLIPModel, CLIPProcessor

print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = clip_model.cuda()
clip_model.eval()
print("CLIP loaded!")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading CLIP model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded!
GPU memory used: 0.61 GB


In [16]:
import torchvision
from torchvision import transforms
from torchvision.datasets import PCAM
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Drive'dan yükle - artık buraya indirdik
PCAM_DIR = '/content/drive/MyDrive/PRISM/datasets/pcam/'

print("Loading PCam dataset...")
train_dataset = PCAM(root=PCAM_DIR, split='train', download=False, transform=transform)
val_dataset   = PCAM(root=PCAM_DIR, split='val',   download=False, transform=transform)
test_dataset  = PCAM(root=PCAM_DIR, split='test',  download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset):,} samples")
print(f"Val:   {len(val_dataset):,} samples")
print(f"Test:  {len(test_dataset):,} samples")

Loading PCam dataset...
Train: 262,144 samples
Val:   32,768 samples
Test:  32,768 samples


In [17]:
import numpy as np
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting features"):
            images = images.to(device)
            outputs = model.vision_model(pixel_values=images)
            features = outputs.pooler_output
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(clip_model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(clip_model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(clip_model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting features: 100%|██████████| 512/512 [45:09<00:00,  5.29s/it]


Train: (262144, 768)
Extracting val features...


Extracting features: 100%|██████████| 64/64 [03:56<00:00,  3.70s/it]


Val: (32768, 768)
Extracting test features...


Extracting features: 100%|██████████| 64/64 [08:15<00:00,  7.75s/it]

Test: (32768, 768)


In [18]:
import os
EMBED_DIR = '/content/drive/MyDrive/PRISM/embeddings/clip/pcam'
os.makedirs(EMBED_DIR, exist_ok=True)

np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy', train_labels)
np.save(f'{EMBED_DIR}/val_features.npy', val_features)
np.save(f'{EMBED_DIR}/val_labels.npy', val_labels)
np.save(f'{EMBED_DIR}/test_features.npy', test_features)
np.save(f'{EMBED_DIR}/test_labels.npy', test_labels)
print(f"Saved to: {EMBED_DIR}")

Saved to: /content/drive/MyDrive/PRISM/embeddings/clip/pcam


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
import pandas as pd

def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() > 0:
            acc  = y_true[mask].mean()
            conf = y_prob[mask].mean()
            ece += mask.sum() * abs(acc - conf)
    return ece / len(y_true)

def run_label_fraction(train_feats, train_lbls, val_feats, val_lbls,
                       test_feats, test_lbls, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_lbls) * fraction)
    idx = np.random.choice(len(train_lbls), n, replace=False)

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_feats[idx], train_lbls[idx])

    y_prob = clf.predict_proba(test_feats)[:, 1]
    y_pred = clf.predict(test_feats)

    return {
        'model': 'CLIP', 'dataset': 'PCam',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_lbls, y_prob),
        'f1':    f1_score(test_lbls, y_pred),
        'brier': brier_score_loss(test_lbls, y_prob),
        'ece':   compute_ece(test_lbls, y_prob),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.8798 | ECE: 0.0778 | Brier: 0.1472
  1% | seed 123 | AUROC: 0.8832 | ECE: 0.0885 | Brier: 0.1474
  1% | seed 456 | AUROC: 0.8803 | ECE: 0.0764 | Brier: 0.1472
  5% | seed 42 | AUROC: 0.8997 | ECE: 0.0544 | Brier: 0.1314
  5% | seed 123 | AUROC: 0.9004 | ECE: 0.0611 | Brier: 0.1317
  5% | seed 456 | AUROC: 0.8972 | ECE: 0.0495 | Brier: 0.1328
  10% | seed 42 | AUROC: 0.9058 | ECE: 0.0461 | Brier: 0.1266
  10% | seed 123 | AUROC: 0.9054 | ECE: 0.0507 | Brier: 0.1272
  10% | seed 456 | AUROC: 0.9053 | ECE: 0.0493 | Brier: 0.1272
  25% | seed 42 | AUROC: 0.9092 | ECE: 0.0411 | Brier: 0.1239
  25% | seed 123 | AUROC: 0.9091 | ECE: 0.0398 | Brier: 0.1238
  25% | seed 456 | AUROC: 0.9108 | ECE: 0.0405 | Brier: 0.1228
  50% | seed 42 | AUROC: 0.9141 | ECE: 0.0408 | Brier: 0.1205
  50% | seed 123 | AUROC: 0.9145 | ECE: 0.0395 | Brier: 0.1201
  50% | seed 456 | AUROC: 0.9122 | ECE: 0.0429 | Brier: 0.1221
  100% | seed 42 | AUROC: 0.9166 | ECE: 0.0406 | Brier: 0.1184
  1

In [20]:
from scipy.optimize import minimize_scalar

def find_temperature(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_feats, train_lbls, val_feats, val_lbls,
                             test_feats, test_lbls, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_lbls) * fraction)
    idx = np.random.choice(len(train_lbls), n, replace=False)

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_feats[idx], train_lbls[idx])

    val_logits = clf.decision_function(val_feats).reshape(-1,1)
    val_logits = np.hstack([-val_logits, val_logits])
    T = find_temperature(val_logits, val_lbls)

    test_prob_raw = clf.predict_proba(test_feats)[:, 1]
    ece_raw       = compute_ece(test_lbls, test_prob_raw)

    test_logits = clf.decision_function(test_feats).reshape(-1,1)
    test_logits = np.hstack([-test_logits, test_logits])
    scaled      = test_logits / T
    exp_s       = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s = (exp_s / exp_s.sum(axis=1, keepdims=True))[:, 1]
    ece_scaled  = compute_ece(test_lbls, test_prob_s)

    return {
        'model': 'CLIP', 'dataset': 'PCam',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc': roc_auc_score(test_lbls, test_prob_raw),
        'brier_raw': brier_score_loss(test_lbls, test_prob_raw),
        'brier_scaled': brier_score_loss(test_lbls, test_prob_s),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           1.2227   0.0809      0.0139           0.0670
0.05           1.5918   0.0550      0.0321           0.0228
0.10           1.7242   0.0487      0.0354           0.0133
0.25           1.9377   0.0405      0.0387           0.0018
0.50           2.0123   0.0411      0.0414          -0.0004
1.00           2.0840   0.0406      0.0402           0.0004


In [21]:
RESULTS_DIR = '/content/drive/MyDrive/PRISM/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

df.to_csv(f'{RESULTS_DIR}/clip_pcam_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/clip_pcam_temperature_scaling.csv', index=False)
print("Results saved!")

Results saved!
